In [21]:
import pandas as pd
import re
from collections import Counter
from normalize import NumberNormalizer


### Работа с данными 

In [22]:
source_data = pd.read_feather("test.f")
source_true_data = pd.read_feather("calibration.f")
print(f"Форма тестого набора предложений {source_data.shape}")
print(f"Форма итогого набора предложений {source_true_data.shape}")

Форма тестого набора предложений (501, 1)
Форма итогого набора предложений (500, 2)


In [23]:
print(list(source_data.columns))
print(list(source_true_data.columns))


['task_text']
['task_text', 'ground_truth']


#### Колонки в двух файлах
Колонка “task_text” содержит целевой фрагмент. Без капитализации, без пунктуации. Возможны ошибки распознавания. Есть в обоих файлах.

Пример: "бюджет где то триста петьдесят тысич рублеи"

Колонка “ground_truth” содержит нормализованный текст. Есть в файле calibration.



In [24]:
first_ten_sentences_tr = [source_true_data.iloc[i]['task_text'] for i in range(10)]
first_ten_sentences_tr


['вам нужно будет перейти в раздел настройки личного кабинета и внести в список все населенные пункты которые вы используете сейчас у вас их может быть один два три четыре пять шесть семь восемь девять помните что если больше четырех то будет требоваться подтверждение значит у вас должно быть разрешение чтобы кто нибудь снял ну типа попросил кого то снять ролик',
 'нажмите на свободную область экрана и вы увидите в нижней части вот такую информацию рядом с объявлением там отображается стоимость просмотра двести один рубль',
 'пятьсот восемьдесят рублей если брать среднее значение то это примерно стоимость упаковки я правильно понял что это десять тетравей а если точнее то где то двеси шестьдесят пять рублей',
 'да процент небольшой но по сравнению с последними тридцатью сутками тут прирост на семь и восемь процентов то есть предыдущий временной отрезок был показатель видимо еще ниже вот',
 'да там чуть иной способ продвижения но примерно двести с лишним',
 'начали развивать всё нормаль

In [25]:
first_ten_sentences_true = [source_true_data.iloc[i]['ground_truth'] for i in range(10)]
first_ten_sentences_true

['вам нужно будет перейти в раздел настройки личного кабинета и внести в список все населенные пункты которые вы используете сейчас у вас их может быть 1 2 3 4 5 6 7 8 9 помните что если больше 4 то будет требоваться подтверждение значит у вас должно быть разрешение чтобы кто нибудь снял ну типа попросил кого то снять ролик',
 'нажмите на свободную область экрана и вы увидите в нижней части вот такую информацию рядом с объявлением там отображается стоимость просмотра 201 рубль',
 '580 рублей если брать среднее значение то это примерно стоимость упаковки я правильно понял что это 10 тетравей а если точнее то где то 265 рублей',
 'да процент небольшой но по сравнению с последними 30 сутками тут прирост на 7 и 8 процентов то есть предыдущий временной отрезок был показатель видимо еще ниже вот',
 'да там чуть иной способ продвижения но примерно 200 с лишним',
 'начали развивать всё нормально им развивать и это крутая история но просто продвижение не должно идти по 37 тире 200 рублей вот в 

In [26]:
first_ten_sentences_test = [source_data.iloc[i]['task_text'] for i in range(10)]
first_ten_sentences_test

['ну да тут выходит примерно на минус несколько процентов соотношение с конкурентами и разница не столь значительная получается за счёт этого он вроде как не влияет если поднимать как я вижу лучше работает именно с тридцати восьми если повыше находишься тогда да тогда точно что оно хорошо будет функционировать',
 'да конечно вы можете активировать и проверить допустим тридцать суток даже можно на ближайший день после активации допустим сделать так чтобы на новый период было бесплатно',
 'можно начать с расширенного варианта за три тысячи или за шесть тысяч с максимального это я просто просила чтобы мне было понятно да как объяснить как настроить ну тут без разницы можно и с телефона и с компа',
 'на какую сумму вы вышлете квитанцию шестнадцать на десять скажите пожалуйста',
 'ну да да да ну смотри сейчас стоимость около сорока восьми да не стоит это того ну вот если вспомнит',
 'максимальный нажим при подключении можно будет через шесть месяцев с вашего кошелька вычтут нужную сумму',
 

In [27]:
def analyze():
    main_df = pd.read_feather('calibration.f')

    list_change = []
    number_patterns = []
    for i,j in main_df.iterrows():
        main = j['task_text']
        upgrate = j['ground_truth']
        if main != upgrate:
            list_change.append(
                {
                    'index' : i,
                    'origin' : main,
                    'upgrate' : upgrate
                }
            )
        #числоввые парметры
        target_numbers = re.findall(r'\b\d+\b', upgrate)
        if target_numbers:
                number_patterns.append({
                    'idx': i,
                    'original': main,
                    'target': upgrate,
                    'numbers': target_numbers
                })
    print(f"Всего примеров с изменениями: {len(list_change)}")
    print(f"Примеров с числами: {len(number_patterns)}")

    word_freq = Counter()
    number_words = [
        'один', 'два', 'три', 'четыре', 'пять', 'шесть', 'семь', 'восемь', 'девять',
        'десять', 'одиннадцать', 'двенадцать', 'тринадцать', 'четырнадцать', 'пятнадцать',
        'шестнадцать', 'семнадцать', 'восемнадцать', 'девятнадцать', 'двадцать',
        'тридцать', 'сорок', 'пятьдесят', 'шестьдесят', 'семьдесят', 'восемьдесят', 'девяносто',
        'сто', 'двести', 'триста', 'четыреста', 'пятьсот', 'шестьсот', 'семьсот', 'восемьсот', 'девятьсот',
        'тысяча', 'тысячи', 'тысяч', 'миллион', 'миллиона', 'миллионов'
    ]
    

    distorted_numbers = Counter()
    
    for change in list_change:
        words = change['origin'].split()
        for word in words:

            if word in number_words:
                word_freq[word] += 1
            

            for num_word in number_words:
                if len(word) >= 3 and len(num_word) >= 3:

                    if (word.startswith(num_word[:3]) or 
                        num_word.startswith(word[:3]) or
                        word.endswith(num_word[-3:]) or
                        num_word.endswith(word[-3:])):
                        if word != num_word:
                            distorted_numbers[f"{word} -> {num_word}"] += 1
    

    for word, count in word_freq.most_common(20):
        print(f"{word}: {count}")
    

    for distortion, count in distorted_numbers.most_common(20):
        print(f"{distortion}: {count}")
    

    
    for pattern in number_patterns[:20]:
        original_words = pattern['original'].split()
        target_words = pattern['target'].split()
        
        print(f"\nПример {pattern['idx']}:")
        print(f"Исходный: {pattern['original']}")
        print(f"Целевой:  {pattern['target']}")
        print(f"Числа: {pattern['numbers']}")
    
    return list_change, number_patterns, word_freq, distorted_numbers


changes, patterns, freq, distortions = analyze()

Всего примеров с изменениями: 494
Примеров с числами: 494
двести: 101
двадцать: 75
пять: 65
пятьдесят: 58
тридцать: 51
тысяч: 46
три: 45
восемьдесят: 38
восемь: 37
триста: 31
два: 30
семь: 27
сорок: 27
один: 26
шесть: 19
сто: 17
десять: 16
четыре: 15
пятьсот: 15
семьдесят: 15
есть -> шесть: 201
двести -> двенадцать: 101
двадцать -> два: 75
двадцать -> одиннадцать: 75
двадцать -> двенадцать: 75
двадцать -> тринадцать: 75
двадцать -> четырнадцать: 75
двадцать -> пятнадцать: 75
двадцать -> шестнадцать: 75
двадцать -> семнадцать: 75
двадцать -> восемнадцать: 75
двадцать -> девятнадцать: 75
двадцать -> тридцать: 75
пять -> девять: 65
пять -> десять: 65
пять -> пятнадцать: 65
пять -> пятьдесят: 65
пять -> пятьсот: 65
двеси -> двенадцать: 65
двеси -> двести: 65

Пример 0:
Исходный: вам нужно будет перейти в раздел настройки личного кабинета и внести в список все населенные пункты которые вы используете сейчас у вас их может быть один два три четыре пять шесть семь восемь девять помните что ес

In [28]:
def generate_final_answer():

    

    df_test = pd.read_feather('test.f')
    print(f"Загружено тестовых примеров: {len(df_test)}")
    

    normalizer = NumberNormalizer()
    
    answers = []
    
    for i, text in enumerate(df_test['task_text']):
        if i % 100 == 0:
            print(f"Обработано: {i}/{len(df_test)}")
        
        normalized = normalizer.normalize_text(text)
        answers.append(normalized)
    
  
    df_test['answer'] = answers
    

    df_test.to_csv('answer.csv', index=False)

    

    for i in range(5):
        print(f"Пример {i+1}:")
        print(f"  Исходный: {df_test.iloc[i]['task_text'][:100]}...")
        print(f"  Результат: {df_test.iloc[i]['answer'][:100]}...")
        print()
    
    return df_test

In [29]:
generate_final_answer()

Загружено тестовых примеров: 501
Обработано: 0/501
Обработано: 100/501
Обработано: 200/501
Обработано: 300/501
Обработано: 400/501
Обработано: 500/501
Пример 1:
  Исходный: ну да тут выходит примерно на минус несколько процентов соотношение с конкурентами и разница не стол...
  Результат: ну да тут выходит примерно на минус несколько процентов соотношение с конкурентами и разница не стол...

Пример 2:
  Исходный: да конечно вы можете активировать и проверить допустим тридцать суток даже можно на ближайший день п...
  Результат: да конечно вы можете активировать и проверить допустим 30 суток даже можно на ближайший день после а...

Пример 3:
  Исходный: можно начать с расширенного варианта за три тысячи или за шесть тысяч с максимального это я просто п...
  Результат: можно начать с расширенного варианта за 3000 или за 6000 с максимального это я просто просила чтобы ...

Пример 4:
  Исходный: на какую сумму вы вышлете квитанцию шестнадцать на десять скажите пожалуйста...
  Результат: на

,task_text,answer
0,ну да тут выходит примерно на минус несколько ...,ну да тут выходит примерно на минус несколько ...
1,да конечно вы можете активировать и проверить ...,да конечно вы можете активировать и проверить ...
2,можно начать с расширенного варианта за три ты...,можно начать с расширенного варианта за 3000 и...
3,на какую сумму вы вышлете квитанцию шестнадцат...,на какую сумму вы вышлете квитанцию 16 на 10 с...
4,ну да да да ну смотри сейчас стоимость около с...,ну да да да ну смотри сейчас стоимость около 4...
...,...,...
496,конечно посмотрим если взглянуть на данные то ...,конечно посмотрим если взглянуть на данные то ...
497,давайте так скажу почти все но я еще встречаю ...,давайте так скажу почти все но я еще встречаю ...
498,в любом случае вы выбираете уровень продвижени...,в любом случае вы выбираете уровень продвижени...
499,я просто сейчас рассматриваю ваши текущие пара...,я просто сейчас рассматриваю ваши текущие пара...


In [30]:
def evaluate():
    
    df = pd.read_feather('calibration.f')
    

    normalizer = NumberNormalizer()
    
    predictions = []
    for text in df['task_text']:
        normalized = normalizer.normalize_text(text)
        predictions.append(normalized)
    
    correct = 0
    total = len(df)
    
    
    errors = []
    for i, (pred, true) in enumerate(zip(predictions, df['ground_truth'])):
        if pred == true:
            correct += 1
        else:
            errors.append({
                'idx': i,
                'predicted': pred,
                'ground_truth': true,
                'original': df.iloc[i]['task_text']
            })
    
    accuracy = correct / total
    print(f"Правильных: {correct}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})")
    

   
 
    return accuracy, errors
evaluate()

Правильных: 408
Accuracy: 0.8160 (81.60%)


(0.816,
 [{'idx': 6,
   'predicted': 'вот и еще по каким причинам я перезваниваю потому что в последний раз могу компенсировать расходы по акциям подскажите успеете использовать бюджет на рекламу там на показы суммарно от 200 до десяти 1000',
   'ground_truth': 'вот и еще по каким причинам я перезваниваю потому что в последний раз могу компенсировать расходы по акциям подскажите успеете использовать бюджет на рекламу там на показы суммарно от 200 до 10000',
   'original': 'вот и еще по каким причинам я перезваниваю потому что в последний раз могу компенсировать расходы по акциям подскажите успеете использовать бюджет на рекламу там на показы суммарно от двухсот до десяти тыщ'},
  {'idx': 15,
   'predicted': 'понятно что мы ее совсем недавно обновили и теперь уже скидку реально как раз настраивать ползунком и там в принципе нет таких вот крайностей когда либо только этот вариант либо соответственно вот он условно минимальный или оптимальный размер скидки у вас теперь появилась возможнос